In [1]:
#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpad\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XdgTimestepping;
Init();

In [2]:
using BoSSS.Foundation.Grid.Classic;
using ilPSP.Utils;
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;
using System.Threading.Tasks;
using BoSSS.Foundation.Grid;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.Control;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using ZwoLevelSetSolver;
using ZwoLevelSetSolver.SolidPhase;
using BoSSS.Solution.XNSECommon;

In [3]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client @C:\Users\miao\Documents\Database
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client MSHPC-AllNodes @DC2, @\\dc1\userspace\miao\cluster"
2,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client MSHPC-AllNodes @DC2, @\\dc1\userspace\miao\cluster"


In [4]:
static var myBatch = ExecutionQueues[0];
static var myDb = myBatch.CreateOrOpenCompatibleDatabase("C:/Users/miao/Documents/Database/Droplet-EE");

In [5]:
BoSSSshell.WorkflowMgm.Init("Droplet-EE");

Project name is set to 'Droplet-EE'.
Default Execution queue is chosen for the database.
Opening existing database '\\dc1\userspace\miao\cluster\Droplet-EE'.


In [6]:
GetDefaultQueue()

RuntimeLocation,win\amd64
DeploymentBaseDirectory,\\dc1\userspace\miao\cluster
DeployRuntime,False
Name,MSHPC-AllNodes
DotnetRuntime,dotnet
Username,FDY\miao
ServerName,DC2
ComputeNodes,"[ hpccluster, hpccluster2, hpcluster3, hpcluster4 ]"
DefaultJobPriority,Normal
SingleNode,True
AllowedDatabasesPaths,[ \\dc1\userspace\miao\cluster ]


In [7]:
BoSSSshell.WorkflowMgm.AllJobs

In [8]:
wmg.DefaultDatabase

{ Session Count = 1; Grid Count = 1; Path = \\dc1\userspace\miao\cluster\Droplet-EE }

In [9]:
databases

#0: { Session Count = 1; Grid Count = 1; Path = \\dc1\userspace\miao\cluster\Droplet-EE }


In [10]:
//static var myDb = OpenOrCreateDatabase(@"C:\Databases\BoSSS_DB");
//BoSSSshell.WorkflowMgm.Init("HeatedSquareCavity");

## Create grid

In [11]:
public static class GridFactory {
 
    public static Grid2D GenerateGrid() { 
        double xSize = 6;
        double yTop = (3.5 - 0.00625);
        double yBottom = (- 2.5 - 0.00625);
        int kelem = 12;

        double[] Xnodes = GenericBlas.Linspace(-xSize, xSize, kelem + 1);
        double[] Ynodes = GenericBlas.Linspace(yBottom, yTop, kelem / 2 + 1);
                var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes, periodicX: true);

                grd.EdgeTagNames.Add(1, "wall_lower");
                grd.EdgeTagNames.Add(2, "pressure_outlet_upper");
                //grd.EdgeTagNames.Add(3, "wall_left");
                //grd.EdgeTagNames.Add(4, "wall_right");

                grd.DefineEdgeTags(delegate (double[] X) {
                    byte et = 0;
                    if(Math.Abs(X[1] - yBottom) <= 1.0e-8)
                        et = 1;
                    if(Math.Abs(X[1] - yTop) <= 1.0e-8)
                        et = 2;
                    //if(Math.Abs(X[0] + xSize) <= 1.0e-8)
                    //    et = 3;
                    //if(Math.Abs(X[0] - xSize) <= 1.0e-8)
                    //   et = 4;

                    return et;
                });

                return grd;
     }
 
 }

In [12]:
public static class BoundaryValueFactory { 
    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
           stw.WriteLine("static class BoundaryValues {");
           stw.WriteLine("  static public double VelX(double[] X) {");
           stw.WriteLine("    return 0.0;");
           stw.WriteLine("  }");

           stw.WriteLine("  static public double VelY(double[] X) {");
           stw.WriteLine("    return 0.046 / 1.778;");
           stw.WriteLine("  }");
           
           stw.WriteLine(" static public double InitialPhi(double[] X) { ");
           stw.WriteLine("    return (((X[0] - 0.0).Pow2() + (X[1] - 0.0).Pow2()).Sqrt() - 1.778);");
           stw.WriteLine("    }"); 
           
           stw.WriteLine(" static public double InitialPhi1(double[] X) { ");
           stw.WriteLine("    return -(X[1] - 0.001);");
           stw.WriteLine("    }"); 

           stw.WriteLine("}"); 
           return stw.ToString();
        }
    }
   
    static public Formula Get_VelX() {
        return new Formula("BoundaryValues.VelX", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_VelY(){
        return new Formula("BoundaryValues.VelY", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InitialPhi(){
        return new Formula("BoundaryValues.InitialPhi", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_InitialPhi1(){
        return new Formula("BoundaryValues.InitialPhi1", AdditionalPrefixCode:GetPrefixCode());
    }
}

## Create Control file

In [13]:
        public static ZLS_Control Aland( int p = 2, int AMRlvl = 0) {
            ZLS_Control C = new ZLS_Control(p);
            //C.ImmediatePlotPeriod = 1;

            //C.SuperSampling = 3;

            C.AgglomerationThreshold = 0.3;
            C.NoOfMultigridLevels = 1;

            int D = 2;

            AppControl._TimesteppingMode compMode = AppControl._TimesteppingMode.Transient;

            //_DbPath = @"\\fdyprime\userspace\smuda\cluster\cluster_db";
            //_DbPath = @"D:\local\local_Testcase_databases\Testcase_ContactLine";
            //_DbPath = @"D:\local\local_spatialConvStudy\StaticDropletOnPlateConvergence\SDoPConvDB";

            // basic database options
            // ======================
            #region db

            C.savetodb = true;
            C.DbPath = @"C:\Users\miao\Documents\Database\Droplet-EE";
            C.ProjectName = "Droplet";
            C.SessionName = "Droplet-check";
            C.ProjectDescription = "Droplet running on pc";

            C.ContinueOnIoError = false;

            //C.LogValues = XNSE_Control.LoggingValues.MovingContactLine;
            //C.PostprocessingModules.Add(new MovingContactLineLogging());

            #endregion

            // Physical Parameters
            // ===================
            #region physics

            double scale = 1e-4;
            C.PhysicalParameters.rho_A = 1 * scale * scale * scale;
            C.PhysicalParameters.rho_B = 1260 * scale * scale * scale;
            C.PhysicalParameters.mu_A = 0.1 * scale ;
            C.PhysicalParameters.mu_B = 1.41 * scale;
            double sigma = 0.046;
            C.PhysicalParameters.Sigma = sigma;

            C.PhysicalParameters.betaS_A = 0.0;
            C.PhysicalParameters.betaS_B = 0.0;

            C.PhysicalParameters.betaL = 0.0;
            C.PhysicalParameters.theta_e = Math.PI / 2.0;

            C.PhysicalParameters.IncludeConvection = true;
            C.PhysicalParameters.Material = false;

            C.Material = new Solid() {
                Density = 1000 * scale * scale * scale,
                Lame2 = 1000 * scale,
                Viscosity = 1 * scale
                //Viscosity = 0 0.05e-4 * scale,
            };

            #endregion

            // grid generation
            // ===============
            #region grid

            C.SetGrid(GridFactory.GenerateGrid());

            //C.AddBoundaryValue("wall_lower", "VelocityX", BoundaryValueFactory.Get_VelX());
            //C.AddBoundaryValue("wall_lower", "VelocityY", BoundaryValueFactory.Get_VelX());
            //C.AddBoundaryValue("pressure_outlet_upper", "Pressure", BoundaryValueFactory.Get_VelX());

            #endregion










            // Initial Values
            // ==============

            C.AddInitialValue("Pressure#A", BoundaryValueFactory.Get_VelY());
            C.AddInitialValue("Pressure#B", BoundaryValueFactory.Get_VelX());
            C.AddInitialValue("Phi", BoundaryValueFactory.Get_InitialPhi());
            C.AddInitialValue(ZwoLevelSetSolver.VariableNames.SolidLevelSetCG, BoundaryValueFactory.Get_InitialPhi1());





            // boundary conditions
            // ===================
            #region BC


            //C.AddBoundaryValue("wall_lower");
            //C.AddBoundaryValue("pressure_outlet_upper");
            //C.AddBoundaryValue("wall_left");
            //C.AddBoundaryValue("wall_right");

            C.AdvancedDiscretizationOptions.GNBC_Localization = NavierSlip_Localization.Bulk;
            C.AdvancedDiscretizationOptions.GNBC_SlipLength = NavierSlip_SlipLength.Prescribed_Beta;
            //C.PhysicalParameters.sliplength = 0.001;

            #endregion

            // misc. solver options
            // ====================
            #region solver


            //C.AdvancedDiscretizationOptions.CellAgglomerationThreshold = 0.2;
            //C.AdvancedDiscretizationOptions.PenaltySafety = 40;
            //C.AdvancedDiscretizationOptions.UseGhostPenalties = true;

            C.NonLinearSolver.MaxSolverIterations = 80;
            C.NonLinearSolver.MinSolverIterations = 2;
            //C.Solver_MaxIterations = 50;
            C.NonLinearSolver.ConvergenceCriterion = 1e-8;
            //C.Solver_ConvergenceCriterion = 1e-8;
            C.LevelSet_ConvergenceCriterion = 1e-12;
            C.NonLinearSolver.Globalization = BoSSS.Solution.AdvancedSolvers.Newton.GlobalizationOption.Dogleg;


            //C.Option_LevelSetEvolution = (compMode == AppControl._TimesteppingMode.Steady) ? LevelSetEvolution.None : LevelSetEvolution.FastMarching;
            //C.EllipticExtVelAlgoControl.solverFactory = () => new ilPSP.LinSolvers.PARDISO.PARDISOSolver();
            //C.EllipticExtVelAlgoControl.IsotropicViscosity = 1e-3;
            //C.fullReInit = false; 

            C.AdvancedDiscretizationOptions.FilterConfiguration = CurvatureAlgorithms.FilterConfiguration.NoFilter;
            C.AdvancedDiscretizationOptions.ViscosityMode = ViscosityMode.FullySymmetric;
            C.AdvancedDiscretizationOptions.SST_isotropicMode = BoSSS.Solution.XNSECommon.SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

            C.AdaptiveMeshRefinement = true;
            C.activeAMRlevelIndicators.Add(new ContactPointRefiner { maxRefinementLevel = 4});
            C.AMR_startUpSweeps = 3;

            #endregion


            // Timestepping
            // ============
            #region time

            //C.CheckJumpConditions = true;

            C.TimeSteppingScheme = TimeSteppingScheme.BDF2;
            C.Timestepper_BDFinit = TimeStepperInit.SingleInit;
            C.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;
            

            C.TimesteppingMode = compMode;
            //double dt = 5e-7;
            double dt = 5e-3;
            C.dtMax = dt;
            C.dtMin = dt;
            C.Endtime = 100;
            C.NoOfTimesteps = 5;
            C.saveperiod = 1;

            #endregion

            return C;
        }

## Send and run jobs

In [14]:
    var C_CTRLFILE = Aland();//Create control file.

Grid Edge Tags changed.


In [15]:
    var JobLocal = C_CTRLFILE.CreateJob();

In [16]:
JobLocal.Status

PreActivation

In [17]:
    JobLocal.NumberOfMPIProcs = 1;
    JobLocal.Activate();
    JobLocal.ShowOutput();
    BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(12*60*60);

Error: System.Reflection.TargetInvocationException: Exception has been thrown by the target of an invocation.
 ---> System.TypeInitializationException: The type initializer for 'Submission#5' threw an exception.
 ---> System.ArgumentException: Expecting a relative path.
   at BoSSS.Application.BoSSSpad.BatchProcessorClient.CreateOrOpenCompatibleDatabase(String dbDir) in C:\Users\miao\Documents\Git\BoSSS-FSI-EE\public\src\L4-application\BoSSSpad\BatchProcessorClient.cs:line 364
   at Submission#5..cctor()
   --- End of inner exception stack trace ---
   --- End of inner exception stack trace ---
   at System.RuntimeFieldHandle.GetValue(RtFieldInfo field, Object instance, RuntimeType fieldType, RuntimeType declaringType, Boolean& domainInitialized)
   at System.Reflection.RtFieldInfo.GetValue(Object obj)
   at Microsoft.CodeAnalysis.Scripting.ScriptVariable.get_Value()
   at Microsoft.DotNet.Interactive.CSharp.CSharpKernel.<>c__DisplayClass13_0.<Microsoft.DotNet.Interactive.IKe

Deploying job Droplet-check ... 
Grid is not in database yet...
Grid successfully saved: 602535ba-46a5-4dea-b123-01422f52e018
Deploying executables and additional files ...
Deployment directory: C:\Users\miao\Documents\Database\Droplet-EE-ZwoLevelSetSolver2023Oct16_212948.129020
copied 46 files.
   written file: control.obj
deployment finished.
Starting mini batch processor in external process...
Started mini batch processor on local machine, process id is 10856.
started.

Starting external console ...
(You may close the new window at any time, the job will continue.)
All jobs finished.


In [18]:
//JobLocal.Stdout

In [19]:
wmg.Sessions

#0: Droplet-EE	Droplet-check	10/16/2023 9:29:51 PM	faef935c...
#1: Droplet-EE	Droplet-check*	10/16/2023 6:53:27 PM	d271d1fa...
#2: Droplet-EE	Droplet-check	10/16/2023 6:15:20 PM	2ce034ed...
#3: Droplet-EE	Droplet-check	10/16/2023 6:10:08 PM	172c66bc...
#4: Droplet-EE	Droplet-check*	10/16/2023 6:05:47 PM	a17d4495...
#5: Droplet-EE	Droplet-check*	10/16/2023 6:03:19 PM	e89edb06...
#6: Droplet-EE	Droplet-check	10/16/2023 5:57:28 PM	3f65162c...
#7: Droplet-EE	Droplet-check	10/16/2023 5:50:48 PM	249c7a68...
#8: Droplet-EE	Droplet-check*	10/16/2023 5:49:16 PM	4478ae96...
#9: Droplet-EE	Droplet-check*	10/16/2023 5:34:45 PM	cadd610b...
#10: Droplet-EE	Droplet-check*	10/16/2023 5:29:40 PM	5c400d18...
#11: Droplet-EE	Droplet-check	10/16/2023 5:26:11 PM	e86af839...
#12: Droplet-EE	Droplet-check	10/16/2023 5:03:56 PM	ed8ade43...
#13: Droplet-EE	Droplet-check	10/16/2023 4:56:16 PM	c38704a3...
#14: Droplet-EE	Droplet-check	10/16/2023 4:53:06 PM	40cf113c...
#15: Droplet-EE	Droplet-check	10/16/2023 4:4

In [20]:
var outPath = wmg.Sessions[0].Export().WithSupersampling(3).Do();

Starting export process... Data will be written to the directory: C:\Users\miao\AppData\Local\BoSSS\plots\sessions\Droplet-EE__Droplet-check__faef935c-7f2f-4338-a03e-4d8227c0432c


## Post processing

In [21]:
//var f = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(1).GetField("Phi");

In [22]:
//var v = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(5).GetField("VelocityX");